# SwinIR SEM 微调 — 准备阶段

GPU: 2×T4 | 框架: BasicSR | 模型: SwinIR-M x4

完成所有 cell 后 `Save and Run All`，确认无报错再正式训练。

In [11]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
        print(f'  GPU {i}: {props.name} — {mem/1e9:.1f} GB')

PyTorch: 2.10.0+cpu
CUDA: False, GPUs: 0


In [12]:
import os
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)
print(f'PWD: {os.getcwd()}')
print(f'Input 目录:')
for d in sorted(os.listdir('/kaggle/input/')):
    print(f'  {d}')
    sub = os.path.join('/kaggle/input/', d)
    if os.path.isdir(sub):
        for f in sorted(os.listdir(sub)):
            fp = os.path.join(sub, f)
            sz = os.path.getsize(fp)/1e6 if os.path.isfile(fp) else '-'
            print(f'    {f} ({sz} MB)' if isinstance(sz, float) else f'    {f}/')

PWD: /kaggle/working
Input 目录:
  datasets
    logdog012/
  models
    logdog012/


In [13]:
if not os.path.exists(os.path.join(WORK_DIR, 'BasicSR')):
    !git clone https://github.com/Log-Dog012/BasicSR.git
    !cd BasicSR && git checkout cuda-sem-finetune
else:
    print('BasicSR 已存在')
!cd BasicSR && git branch -v

BasicSR 已存在
* cuda-sem-finetune 6034ccf feat: Kaggle 训练准备 notebook


In [14]:
os.chdir(os.path.join(WORK_DIR, 'BasicSR'))
!pip install -r requirements.txt -q
!pip install -e . -q
!pip install lpips timm -q
print('✅ 依赖安装完成')
!pip list 2>/dev/null | grep -E 'basicsr|lpips|timm|torch '

  Preparing metadata (setup.py) ... done
✅ 依赖安装完成
basicsr                                  1.4.2               /kaggle/working/BasicSR
lpips                                    0.1.4
timm                                     1.0.26
torch                                    2.10.0+cpu


## 复制模型文件

从 Kaggle Model Input (`logdog012/swinir-finetune`) 复制到 BasicSR 目录结构中。

In [15]:
import shutil

# Kaggle Model 挂载路径（已从目录.ipynb 确认）
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/1'

print(f'模型目录: {MODEL_INPUT}')
if os.path.exists(MODEL_INPUT):
    for f in sorted(os.listdir(MODEL_INPUT)):
        full = os.path.join(MODEL_INPUT, f)
        if os.path.isfile(full):
            print(f'  {f} ({os.path.getsize(full)/1e6:.1f}MB)')
        else:
            print(f'  {f}/ ({len(os.listdir(full))} items)')

模型目录: /kaggle/input/models/logdog012/swinir-finetune/pytorch/default/1
  finetune_SwinIR_SRx4_SEM/ (1 items)
  pretrained_models/ (1 items)


In [16]:
# 复制预训练权重 → SwinIR/model_zoo/
swinir_zoo = os.path.join(WORK_DIR, 'BasicSR', 'SwinIR', 'model_zoo')
pretrained_name = '001_classicalSR_DIV2K_s48w8_SwinIR-M_x4.pth'

found = False
for root, dirs, files in os.walk(MODEL_INPUT):
    for f in files:
        if pretrained_name in f:
            os.makedirs(swinir_zoo, exist_ok=True)
            shutil.copy2(os.path.join(root, f), os.path.join(swinir_zoo, pretrained_name))
            print(f'✅ 预训练权重: {os.path.getsize(os.path.join(swinir_zoo, pretrained_name))/1e6:.1f}MB')
            found = True
            break
    if found: break
if not found:
    print(f'⚠️ 未找到 {pretrained_name}')

✅ 预训练权重: 59.6MB


In [17]:
# 复制 checkpoint → experiments/
exp_dir = os.path.join(WORK_DIR, 'BasicSR', 'experiments', 'finetune_SwinIR_SRx4_SEM')
models_dir = os.path.join(exp_dir, 'models')
states_dir = os.path.join(exp_dir, 'training_states')
os.makedirs(models_dir, exist_ok=True)
os.makedirs(states_dir, exist_ok=True)

for src_name, dst_dir in [('net_g_10000.pth', models_dir), ('10000.state', states_dir)]:
    found = False
    for root, dirs, files in os.walk(MODEL_INPUT):
        for f in files:
            if src_name == f:
                shutil.copy2(os.path.join(root, f), os.path.join(dst_dir, src_name))
                print(f'✅ {src_name}: {os.path.getsize(os.path.join(dst_dir, src_name))/1e6:.1f}MB')
                found = True
                break
        if found: break
    if not found:
        print(f'⚠️ 未找到 {src_name}')

✅ net_g_10000.pth: 119.2MB
✅ 10000.state: 95.6MB


## 调整训练配置路径

In [18]:
import yaml

cfg_path = os.path.join(WORK_DIR, 'BasicSR', 'options', 'train', 'SwinIR', 'finetune_SwinIR_SRx4_SEM.yml')
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Kaggle Dataset 挂载路径（已从目录.ipynb 确认）
# 注意: zip 解压后多了一层目录（train_sem_hr/HR 而不是直接 HR）
DATA_INPUT = '/kaggle/input/datasets/logdog012/swinir-finetune'

cfg['datasets']['train']['dataroot_gt'] = f'{DATA_INPUT}/train_sem_hr/HR'
cfg['datasets']['train']['dataroot_lq'] = f'{DATA_INPUT}/train_sem_lr_x4/LR_x4'
cfg['datasets']['val']['dataroot_gt'] = f'{DATA_INPUT}/eval_sem/eval/SEMimg'
cfg['datasets']['val']['dataroot_lq'] = f'{DATA_INPUT}/eval_sem/eval/LR'
cfg['path']['pretrain_network_g'] = os.path.abspath(os.path.join(swinir_zoo, pretrained_name))
cfg['num_gpu'] = 2
cfg['datasets']['train']['batch_size_per_gpu'] = 8

with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print('配置已更新:')
print(f'  训练: {cfg["datasets"]["train"]["dataroot_gt"]}')
print(f'  验证: {cfg["datasets"]["val"]["dataroot_gt"]}')
print(f'  num_gpu={cfg["num_gpu"]}, batch={cfg["datasets"]["train"]["batch_size_per_gpu"]}')
print(f'  auto_resume={cfg.get("auto_resume", False)}')

配置已更新:
  训练: /kaggle/input/datasets/logdog012/swinir-finetune/train_sem_hr/HR
  验证: /kaggle/input/datasets/logdog012/swinir-finetune/eval_sem/eval/SEMimg
  num_gpu=2, batch=8
  auto_resume=True


## 目录结构检查

In [19]:
print('目录检查:')
for label, path in [
    ('训练 HR', cfg['datasets']['train']['dataroot_gt']),
    ('训练 LR', cfg['datasets']['train']['dataroot_lq']),
    ('验证 HR', cfg['datasets']['val']['dataroot_gt']),
    ('验证 LR', cfg['datasets']['val']['dataroot_lq']),
    ('模型权重', f'{models_dir}/net_g_10000.pth'),
    ('训练状态', f'{states_dir}/10000.state'),
    ('预训练', f'{swinir_zoo}/{pretrained_name}'),
]:
    ok = os.path.exists(path)
    extra = ''
    if ok and os.path.isfile(path):
        extra = f' ({os.path.getsize(path)/1e6:.1f}MB)'
    elif ok:
        extra = f' ({len(os.listdir(path))} items)'
    print(f'  {"✅" if ok else "❌"} {label}{extra}')

目录检查:
  ✅ 训练 HR (19256 items)
  ✅ 训练 LR (19256 items)
  ✅ 验证 HR (210 items)
  ✅ 验证 LR (210 items)
  ✅ 模型权重 (119.2MB)
  ✅ 训练状态 (95.6MB)
  ✅ 预训练 (59.6MB)


## Checkpoint 加载测试 (CPU)

In [20]:
import torch
from basicsr.archs.swinir_arch import SwinIR

print('=== Checkpoint 加载测试 ===')

# 加载训练状态
state = torch.load(f'{states_dir}/10000.state', map_location='cpu', weights_only=False)
print(f'状态: iter={state["iter"]}, epoch={state["epoch"]}')
# optimizers 可能是 list 或 dict
opt = state.get("optimizers", [])
print(f'  optimizers: {list(opt.keys()) if isinstance(opt, dict) else f"list[{len(opt)}]"}')
sche = state.get("schedulers", [])
print(f'  schedulers: {list(sche.keys()) if isinstance(sche, dict) else f"list[{len(sche)}]"}')

# 加载模型权重
model_dict = torch.load(f'{models_dir}/net_g_10000.pth', map_location='cpu', weights_only=False)
print(f'权重: keys={list(model_dict.keys())}')

# 构建模型并加载
model = SwinIR(
    upscale=4, in_chans=3, img_size=48, window_size=8,
    img_range=1., depths=[6,6,6,6,6,6], embed_dim=180,
    num_heads=[6,6,6,6,6,6], mlp_ratio=2,
    upsampler='pixelshuffle', resi_connection='1conv'
)
model.load_state_dict(model_dict['params_ema'], strict=True)
n_params = sum(p.numel() for p in model.parameters())
print(f'模型: 参数量={n_params:,}')

# CPU 前向传播
model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 64, 64))
print(f'前向传播: (1,3,64,64) -> {list(out.shape)}')
print('\n✅ 所有检查通过！')

=== Checkpoint 加载测试 ===
状态: iter=10000, epoch=2
  optimizers: list[1]
  schedulers: list[1]
权重: keys=['params', 'params_ema']


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


模型: 参数量=11,900,199
前向传播: (1,3,64,64) -> [1, 3, 256, 256]

✅ 所有检查通过！


## 完成

全部 ✅ 后：
1. `Save and Run All`
2. 下载 output 检查目录
3. 无误 → 新 notebook 正式训练 (GPU 2×T4)